<a href="https://colab.research.google.com/github/dtoralg/TheValley_MDS/blob/main/%5B01%5D%20-%20Intro_No_Supervisados/%5B01%5D%20-%20Notebooks/E1_Distancias_y_Efectos_de_Escala.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E1 · Distancias y efectos de escala - Introducción a los modelos no supervisados

## Introducción

> *Sin etiquetas, buscamos estructura en los datos.*

En no supervisado no hay target: queremos **agrupar lo parecido**. Pero "parecido" hay que
medirlo con números, y para eso usamos una **distancia**. En este ejercicio:

1. Calculamos distancias **numéricas**: Euclídea, Manhattan y Minkowski.
2. Vemos el **efecto de la escala**: sin escalar, la variable más grande manda.
3. Calculamos distancias para datos **binarios**: Jaccard y coeficiente simple.
4. Entrenamos un K-Means base **con y sin escalar** para ver el efecto en la práctica.

## Objetivos del ejercicio

- Entender la **distancia** como forma de medir el parecido entre observaciones.
- Calcular distancias Euclídea, Manhattan y Minkowski, y las binarias Jaccard y coef. simple.
- Comprobar **por qué escalar es obligatorio** en clustering.

## Descripción del dataset (clientes sin etiqueta)

Imagina que tienes una base de clientes y quieres ofrecerles un descuento, pero **no hay
etiquetas**: nadie te ha dicho qué cliente es de qué tipo. El objetivo del aprendizaje no
supervisado es justo ese: **descubrir la estructura** que hay dentro de los datos.

Generamos un dataset **sintético y reproducible** con `generar_clientes` (autocontenido en
Colab). Cada fila es un cliente con estas variables:

| Variable | Tipo | Descripción |
|---|---|---|
| `gasto_anual` | numérica | Gasto total al año (€), escala de miles |
| `num_visitas` | numérica | Nº de visitas al año, escala de decenas |
| `ticket_medio` | numérica | Gasto medio por compra (€) |
| `antiguedad_meses` | numérica | Meses como cliente |
| `edad` | numérica | Edad del cliente |
| `usa_app` | binaria | 1 si usa la app |
| `tiene_tarjeta_fidelidad` | binaria | 1 si tiene tarjeta de fidelidad |
| `compra_online` | binaria | 1 si compra online |
| `recibe_newsletter` | binaria | 1 si recibe la newsletter |
| `devuelve_productos` | binaria | 1 si suele devolver productos |

> Fíjate en las **escalas tan distintas** (gasto en miles, visitas en decenas). Esto va a ser
> clave: en clustering, la distancia depende de la escala, así que **habrá que escalar**.

### 1. Importar librerías necesarias

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial import distance
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

pd.set_option("display.width", 120)

### 2. Generar los datos

In [ ]:
import numpy as np
import pandas as pd

def generar_clientes(n=600, semilla=42):
    # Dataset sintetico y reproducible de clientes SIN ETIQUETA para segmentacion.
    # Por dentro hay 4 perfiles latentes que el modelo deberia redescubrir, pero NO los
    # exponemos: en aprendizaje no supervisado no hay target, solo buscamos estructura.
    rng = np.random.default_rng(semilla)
    # perfil: (gasto_anual, num_visitas, ticket_medio, antiguedad_meses, edad,
    #          p_app, p_fidelidad, p_online, p_newsletter, p_devuelve)
    perfiles = [
        (9000, 42, 230, 60, 46, 0.85, 0.90, 0.70, 0.60, 0.10),  # grandes clientes
        (1100,  6, 120, 22, 37, 0.40, 0.20, 0.55, 0.30, 0.10),  # ocasionales
        (3200, 36,  75, 44, 52, 0.50, 0.65, 0.60, 0.80, 0.55),  # cazaofertas
        (2400, 15, 165,  9, 30, 0.92, 0.40, 0.95, 0.50, 0.20),  # nuevos digitales
    ]
    pesos = [0.22, 0.33, 0.25, 0.20]
    seg = rng.choice(len(perfiles), size=n, p=pesos)

    filas = []
    for s in seg:
        g, v, t, a, e, pa, pf, po, pn, pdv = perfiles[s]
        filas.append([
            round(max(50, rng.normal(g, g * 0.22)), 2),    # gasto_anual (€)
            int(max(1, round(rng.normal(v, v * 0.30)))),    # num_visitas
            round(max(5, rng.normal(t, t * 0.22)), 2),      # ticket_medio (€)
            int(max(1, round(rng.normal(a, 12)))),          # antiguedad_meses
            int(np.clip(rng.normal(e, 8), 18, 85)),         # edad
            int(rng.random() < pa),                          # usa_app
            int(rng.random() < pf),                          # tiene_tarjeta_fidelidad
            int(rng.random() < po),                          # compra_online
            int(rng.random() < pn),                          # recibe_newsletter
            int(rng.random() < pdv),                         # devuelve_productos
        ])
    cols = ["gasto_anual", "num_visitas", "ticket_medio", "antiguedad_meses", "edad",
            "usa_app", "tiene_tarjeta_fidelidad", "compra_online", "recibe_newsletter",
            "devuelve_productos"]
    return pd.DataFrame(filas, columns=cols)

In [ ]:
df = generar_clientes(n=600, semilla=42)
print("Clientes:", df.shape[0], "| variables:", df.shape[1])
df.head()

In [ ]:
# Fijate en las escalas tan distintas (gasto en miles, visitas en decenas)
df[["gasto_anual", "num_visitas", "ticket_medio", "edad"]].describe().round(1)

### 3. Distancias numéricas

Tomamos dos clientes y medimos lo "lejos" que están con tres distancias:

- **Euclídea**: la distancia geométrica en línea recta.
- **Manhattan**: suma de diferencias absolutas (movimiento en cuadrícula).
- **Minkowski**: generaliza las dos anteriores con un parámetro `p` (p=2 es Euclídea, p=1 es Manhattan).

In [ ]:
num_cols = ["gasto_anual", "num_visitas", "ticket_medio", "antiguedad_meses", "edad"]
a = df.loc[0, num_cols].to_numpy(dtype=float)
b = df.loc[1, num_cols].to_numpy(dtype=float)
print("Cliente A:", a)
print("Cliente B:", b)
print()
print(f"Euclídea  (p=2): {distance.euclidean(a, b):.2f}")
print(f"Manhattan (p=1): {distance.cityblock(a, b):.2f}")
print(f"Minkowski (p=3): {distance.minkowski(a, b, p=3):.2f}")

### 4. El efecto de la escala

¿Tratan estas distancias por igual a `gasto_anual` (miles) y a `num_visitas` (decenas)? No:
sin escalar, la variable más grande **domina** la distancia. Veamos cuánto aporta cada
variable a la distancia Euclídea (al cuadrado) entre A y B:

In [ ]:
contrib = pd.Series((a - b) ** 2, index=num_cols)
print("Aporte de cada variable a la distancia (sin escalar):")
print((contrib / contrib.sum() * 100).round(1).astype(str) + " %")
print("\nCasi toda la distancia la decide 'gasto_anual' solo por su escala.")

In [ ]:
# Escalamos (media 0, desviación 1) y repetimos
scaler = StandardScaler().fit(df[num_cols])
A_esc = scaler.transform([a])[0]
B_esc = scaler.transform([b])[0]
contrib_esc = pd.Series((A_esc - B_esc) ** 2, index=num_cols)
print("Aporte de cada variable a la distancia (escalando):")
print((contrib_esc / contrib_esc.sum() * 100).round(1).astype(str) + " %")
print("\nAhora todas las variables pueden competir: ninguna manda solo por su tamaño.")

### 5. Distancias para datos binarios

Con variables 0/1 (usa app, tarjeta, online...) usamos otras medidas:

- **Jaccard**: proporción de coincidencias **positivas** sobre las que al menos uno tiene a 1.
  `Jaccard = (ambos 1) / (al menos uno 1)`.
- **Coeficiente simple**: cuenta también las coincidencias en 0.
  `Simple = (ambos 1 + ambos 0) / total`.

In [ ]:
bin_cols = ["usa_app", "tiene_tarjeta_fidelidad", "compra_online", "recibe_newsletter", "devuelve_productos"]
u = df.loc[0, bin_cols].to_numpy(dtype=int)
w = df.loc[5, bin_cols].to_numpy(dtype=int)
print("Cliente A:", dict(zip(bin_cols, u)))
print("Cliente F:", dict(zip(bin_cols, w)))

ambos_1 = int(np.sum((u == 1) & (w == 1)))
ambos_0 = int(np.sum((u == 0) & (w == 0)))
al_menos_uno_1 = int(np.sum((u == 1) | (w == 1)))
n_attr = len(bin_cols)

jaccard = ambos_1 / al_menos_uno_1 if al_menos_uno_1 else 0
simple = (ambos_1 + ambos_0) / n_attr
print(f"\nJaccard = {ambos_1}/{al_menos_uno_1} = {jaccard:.0%}")
print(f"Coef. simple = ({ambos_1}+{ambos_0})/{n_attr} = {simple:.0%}")
# Comprobacion con scipy (devuelve DISimilitud, por eso 1 - dist)
print(f"\nComprobación scipy (similitud Jaccard): {1 - distance.jaccard(u, w):.0%}")

### 6. Un modelo base: ¿escalar o no?

Entrenamos un K-Means sencillo (K=3) sobre las variables numéricas **sin escalar** y
**escalando**. Atención a una trampa: sin escalar, el silhouette puede salir incluso **más
alto**, pero no porque agrupe mejor, sino porque los grupos **colapsan sobre `gasto_anual`**
(separar en una sola dimensión es muy fácil). La prueba está en el perfil: sin escalar, los
clusters solo se distinguen en el gasto e **ignoran** el resto de variables.

In [ ]:
X = df[num_cols]

# Sin escalar
km_raw = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X)
sil_raw = silhouette_score(X, km_raw.labels_)

# Escalando
X_esc = StandardScaler().fit_transform(X)
km_esc = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X_esc)
sil_esc = silhouette_score(X_esc, km_esc.labels_)

print(f"Silhouette SIN escalar: {sil_raw:.3f}  (engañoso: separa casi solo por gasto)")
print(f"Silhouette escalando:   {sil_esc:.3f}")
print("\nPerfil de los grupos SIN escalar (medias por variable):")
print(X.assign(cluster=km_raw.labels_).groupby("cluster").mean().round(0))
print("\n-> Sin escalar, los clusters solo cambian en 'gasto_anual'; el resto queda ignorado.")
print("   Escalando, los grupos usan TODAS las variables: una segmentación útil de verdad.")

### Reflexión

1. ¿Por qué, sin escalar, casi toda la distancia la decide `gasto_anual`?
2. ¿En qué se diferencian Jaccard y el coeficiente simple? ¿Cuándo usarías cada uno?
3. ¿Qué le pasa a los grupos de K-Means cuando no escalas?
4. ¿Por qué decimos que en clustering escalar es "parte obligatoria del proceso"?